# synthesis

information loss against cost, across every experiment.

In [1]:
from pathlib import Path

import pandas as pd

from causality_bench.provenance import read_csv

pd.set_option("display.width", 200)

# results/ sits beside notebooks/ in the repo root
RESULTS = Path.cwd().parent / "results"


def raw(name):
    return read_csv(RESULTS / "raw" / f"{name}.csv")


def processed(name):
    return read_csv(RESULTS / "processed" / f"{name}.csv")


the false-ordering rate and the concurrency ratio side by side, across every experiment.

In [2]:
frames = []
for name in ["baseline", "scaling", "message_rate", "delay", "topology", "failure"]:
    frame = raw(name)
    frame["experiment"] = name
    frames.append(frame)

runs = pd.concat(frames, ignore_index=True)
runs["misrepresented_share"] = runs.lamport_false_orderings / runs.total_pairs
runs["vector_density"] = runs.mean_vector_nonzero_entries / runs.nodes
print("total runs:", len(runs))
runs.groupby("experiment")[["lamport_false_ordering_rate", "misrepresented_share", "vector_density"]].agg(
    ["min", "mean", "max"]
).round(4)

total runs: 1320


lamport_false_ordering_rate                 misrepresented_share                 vector_density                
                                     min    mean     max                  min    mean     max            min    mean     max
experiment                                                                                                                  
baseline                          0.9825  0.9831  0.9839               0.0325  0.0336  0.0350         0.9762  0.9831  0.9864
delay                             0.9718  0.9834  0.9945               0.0169  0.0497  0.1246         0.9370  0.9757  0.9927
failure                           0.9825  0.9856  0.9899               0.0322  0.0372  0.0478         0.9667  0.9805  0.9864
message_rate                      0.9757  0.9830  0.9937               0.0148  0.0434  0.1156         0.9194  0.9783  0.9951
scaling                           0.9490  0.9820  0.9929               0.0066  0.0460  0.0841         0.9563  0.9770  0.9986
topology                          0.9895  0.9943  0.9993               0.0581  0.1216  0.2168         0.8609  0.9350  0.9740

the trade-off in one table: the bytes per message a scalar clock saves, against the share of concurrent pairs it misorders to save them.

In [3]:
cost = processed("scaling_summary")[["nodes", "vector_timestamp_bytes_mean"]].copy()
loss = processed("scaling_summary")[["nodes", "lamport_false_ordering_rate_mean"]]
trade = cost.merge(loss, on="nodes")
trade["bytes_per_message_saved"] = trade.vector_timestamp_bytes_mean - 8
trade

,nodes,vector_timestamp_bytes_mean,lamport_false_ordering_rate_mean,bytes_per_message_saved
0,2,16.0,0.954390,8.0
1,4,32.0,0.975359,24.0
2,8,64.0,0.983078,56.0
3,16,128.0,0.987281,120.0
4,32,256.0,0.989809,248.0
5,64,512.0,0.991540,504.0
6,128,1024.0,0.992751,1016.0
